<a href="https://colab.research.google.com/github/Juhil3001/jenya-practical-test/blob/main/LSTM_Text_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import re
import string
import random

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

print("Tensorflow Version : ", tf.__version__)

print("GPU Devices : ")
print(tf.config.list_physical_devices("GPU"))

Tensorflow Version :  2.20.0
GPU Devices : 
[]


In [4]:
dataset_path = "/content/sakespeare.txt"

if os.path.exists(dataset_path):
  print("Dataset found successfully.")

  file_size = os.path.getsize(dataset_path)
  print("File Size : ", file_size, "bytes")
else:
  print("Error : Dataset not found")

Dataset found successfully.
File Size :  5638480 bytes


In [5]:
with open(dataset_path, "r", encoding="utf-8") as f:
  raw_text = f.read()

print("Dataset Loaded Successfully")
print("Total numbers of character : ", len(raw_text))

print(raw_text[:500])

Dataset Loaded Successfully
Total numbers of character :  5378701
The Project Gutenberg eBook of The Complete Works of William Shakespeare
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
befo


In [6]:
text = raw_text.lower()

text = text.translate(
    str.maketrans("", "", string.punctuation)
)

text = text.strip()

print(text[:1000])

the project gutenberg ebook of the complete works of william shakespeare
    
this ebook is for the use of anyone anywhere in the united states and
most other parts of the world at no cost and with almost no restrictions
whatsoever you may copy it give it away or reuse it under the terms
of the project gutenberg license included with this ebook or online
at wwwgutenbergorg if you are not located in the united states
you will have to check the laws of the country where you are located
before using this ebook

title the complete works of william shakespeare

author william shakespeare


        
release date january 1 1994 ebook 100
                most recently updated august 24 2025

language english

other information and formats wwwgutenbergorgebooks100


 start of the project gutenberg ebook the complete works of william shakespeare 




the complete works of william shakespeare

by william shakespeare




                    contents

    the sonnets
    all’s well that ends well
 

In [7]:
tokenizer = Tokenizer(
    oov_token = "<OOV>"
)

tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1

print("Tokenizer completed.")
print("Total vocabulary size : ", total_words)

for word, index in list(tokenizer.word_index.items())[:30]:
  print(f"{word:20} --> {index}")

Tokenizer completed.
Total vocabulary size :  33516
<OOV>                --> 1
the                  --> 2
and                  --> 3
i                    --> 4
to                   --> 5
of                   --> 6
a                    --> 7
you                  --> 8
my                   --> 9
in                   --> 10
that                 --> 11
is                   --> 12
not                  --> 13
with                 --> 14
me                   --> 15
it                   --> 16
for                  --> 17
his                  --> 18
be                   --> 19
this                 --> 20
your                 --> 21
he                   --> 22
but                  --> 23
have                 --> 24
as                   --> 25
thou                 --> 26
him                  --> 27
so                   --> 28
will                 --> 29
what                 --> 30


In [8]:
token_ids = tokenizer.texts_to_sequences(
    [text]
)[0]

print("Text converted to token IDs successfully.")

print("Total token : ", len(token_ids))

print(token_ids[:50])

Text converted to token IDs successfully.
Total token :  966408
[2, 1090, 1486, 4804, 6, 2, 3199, 1487, 6, 1152, 5988, 20, 4804, 12, 17, 2, 304, 6, 4399, 6431, 10, 2, 3633, 2528, 3, 108, 158, 878, 6, 2, 177, 51, 35, 1729, 3, 14, 637, 35, 13356, 4027, 8, 80, 3200, 16, 99, 16, 134, 53, 13357, 16]


In [9]:
sequence_length = 20

input_sequence = []

for i in range(sequence_length, len(token_ids)):
  sequence = token_ids[i - sequence_length : i + 1]

  input_sequence.append(sequence)

input_sequence = np.array(input_sequence,dtype=np.int32)

print("Total input sequence : ", len(input_sequence))
print("Sequence Shape", input_sequence.shape)

Total input sequence :  966388
Sequence Shape (966388, 21)


In [10]:
x = input_sequence[:, :-1]
y = input_sequence[:, -1]

print("x Shape : ", x.shape)
print("y Shape : ", y.shape)

print("Example of input token IDs : ", x[0])
print("Example of target token IDs : ", y[0])

x Shape :  (966388, 20)
y Shape :  (966388,)
Example of input token IDs :  [   2 1090 1486 4804    6    2 3199 1487    6 1152 5988   20 4804   12
   17    2  304    6 4399 6431]
Example of target token IDs :  10


In [11]:
index_to_word = {
    index : word
    for word, index in tokenizer.word_index.items()
}

example_word = [
    index_to_word.get(token_ids, "<UNKOWN>")
    for token_ids in x[0]
]

target_word = index_to_word.get(
    y[0].item(), # Use .item() to extract the scalar value
    "<UNKOWN>"
)

print("Example input word : ", " ".join(example_word))
print("Example target word : ", target_word)

Example input word :  the project gutenberg ebook of the complete works of william shakespeare this ebook is for the use of anyone anywhere
Example target word :  in


In [12]:
split_index = int(len(x) * 0.90)

x_train = x[:split_index]
y_train = y[:split_index]

x_test = x[split_index:]
y_test = y[split_index:]

print("Total training data : ", len(x_train))
print("Total training data : ", len(y_train))
print("Total testing data : ", len(x_test))
print("Total testing data : ", len(y_test))

Total training data :  869749
Total training data :  869749
Total testing data :  96639
Total testing data :  96639


In [13]:
from tensorflow.keras.layers import Dense, LSTM

embedding_dim = 128
lstm_units = 256

model = Sequential([
    Embedding(
        input_dim = total_words,
        output_dim = embedding_dim
    ),

    LSTM(
        lstm_units
    ),

    Dense(
        total_words,
        activation = "softmax"
    )
])

model.build(
    input_shape = (None, sequence_length)
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │     4,290,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 33516)          │     8,613,612 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,297,900 (50.73 MB)

 Trainable params: 13,297,900 (50.73 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

print("Model compiled successfully")

Model compiled successfully


In [15]:
early_stopping = EarlyStopping(
    monitor = "val_loss",
    patience = 3,
    restore_best_weights = True
)

checkpoint = ModelCheckpoint(
    "best_lstm_model.keras",
    monitor = "val_loss",
    save_best_only = True
)

print("Early stopping configuration.")
print("Model checkpoint configuration.")

Early stopping configuration.
Model checkpoint configuration.


In [ ]:
epochs = 20
batch_size = 5000

history = model.fit(
    x_train, y_train,
    validation_data = (x_test, y_test),
    epochs = epochs,
    batch_size = batch_size,
    callbacks = [early_stopping, checkpoint]
)

Epoch 1/20
174/174 ━━━━━━━━━━━━━━━━━━━━ 3097s 18s/step - accuracy: 0.0315 - loss: 7.1940 - val_accuracy: 0.0282 - val_loss: 7.4663
Epoch 2/20
 52/174 ━━━━━━━━━━━━━━━━━━━━ 29:29 15s/step - accuracy: 0.0316 - loss: 7.0250

In [ ]:
epochs_completed = len(history.history["loss"])

print("Training completed")
print("Epochs actually completed : ", epochs_completed)

In [ ]:
validation_loss, validation_accuracy = model.evaluate(
    x_test, y_test,
    batch_size = batch_size,
)

print("Validation Loss : ", validation_loss)
print("Validation Accuracy : ", validation_accuracy)

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history["loss"], label = "Training Loss")

plt.plot(history.history["val_loss"], label = "Validation Loss")

plt.title("Training VS Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(history.history["accuracy"], label = "Training accuracy")

plt.plot(history.history["val_accuracy"], label = "Validation accuracy")

plt.title("Training VS Validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

In [ ]:
index_to_word = {
    index: word
    for word, index in tokenizer_index.items()
}

print("Reverse vocabulary created")

print("Token ID -->", index_to_word.get(1, "<UNKNOWN>"))

In [ ]:
def generate_text(
    seed_text,
    next_words = 100,
    temerature = 1.0
):
  generated_text = seed_text

  for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences(
        [generated_text]
    )[0]

    token_list = token_list[token_list-sequence_length:]

    token_list = np.array(
        token_list,
        dtype = np.int32
    ).reshape(1, -1)

    predictions = model.predicts(
        token_list,
        verbose=0
    )[0]

    predictions = np.asarray(
        predictions
    ).astype("float64")

    predictions = np.log(
        predictions + 1e-8
    ) / temperature

    probabilities = np.exp(
        predictions
    )

    propbabilities /= np.sum(
        probabilities
    )

    predicted_id = np.random.choice(
        len(probabilities),
        p=probabilities
    )

    predicted_word = index_to_word.get(
        predicted_id,
        ""
    )

    if predicted_word:
      generated_text += " " + predictedd_word

  return generated_text

In [ ]:
seed_text = "to be or not to"

generated_text = generate_text(
    seed_text = seed_text,
    next_words = 20,
    temerature = 0.8
)

print(seed_text)

print(generated_text)

In [ ]:
seeds = [
    "to be or not to",
    "romeo and juliet",
    "king of england"
]

generated_results = []

for seed in seeds:
  generated = generated_text(
      seed_text = seed,
      next_words = 50,
      temerature = 0.8
  )

  generated_results.append(
      {
          "seed" : seed,
          "generated_text" : generated
      }
  )

  print("=" * 80)
  print("Seed : ")
  print(seed)

  print(generated)

  print()